In [ ]:
!wget https://www.lamsade.dauphine.fr/~cazenave/project2026.zip
!unzip project2026.zip
!ls -l

--2026-03-31 14:26:42--  https://www.lamsade.dauphine.fr/~cazenave/project2026.zip
Resolving www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)... 193.48.71.250
Connecting to www.lamsade.dauphine.fr (www.lamsade.dauphine.fr)|193.48.71.250|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 138578548 (132M) [application/zip]
Saving to: ‘project2026.zip’

project2026.zip     100%[===================>] 132.16M  28.1MB/s    in 5.4s    

2026-03-31 14:26:48 (24.6 MB/s) - ‘project2026.zip’ saved [138578548/138578548]

Archive:  project2026.zip
  inflating: games.data              
  inflating: golois.cpython-312-x86_64-linux-gnu.so  
total 665400
-rw-r--r-- 1 root root 542497580 Oct  7  2022 games.data
-rwxr-xr-x 1 root root    284672 Oct  1 15:09 golois.cpython-312-x86_64-linux-gnu.so
-rw-r--r-- 1 root root 138578548 Oct  1 20:02 project2026.zip
drwxr-xr-x 1 root root      4096 Mar 23 13:34 sample_data


In [ ]:
!pip install wandb weave

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 955.0/955.0 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.9/45.9 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 89.9/89.9 kB 11.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.7/1.7 MB 65.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.9/52.9 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.3/207.3 kB 24.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 149.7 MB/s eta 0:00:00


In [ ]:
import wandb

wandb.login(key="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")

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: mohamedzouad9 (mohamedzouad9-psl) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True



```
# This is formatted as code
```

# Base CODE

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE BASE — Infrastructure partagée | Projet Go 19×19
#  À exécuter UNE SEULE FOIS avant n'importe quel modèle
# ═══════════════════════════════════════════════════════════════════

import os
import gc
import csv
import random
import shutil
import datetime
import numpy as np
import tensorflow as tf
import tensorflow.keras as keras
from tensorflow.keras import layers, regularizers
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt

# ── Juste après les imports ───────────────────────────────────────

# ← AJOUTÉ : mixed precision — FP16 sur Tensor Cores (T4 / P100)
keras.mixed_precision.set_global_policy('float32')

# ← AJOUTÉ : détection multi-GPU
strategy = tf.distribute.MirroredStrategy()
NUM_GPUS  = strategy.num_replicas_in_sync
print(f"GPUs détectés : {NUM_GPUS}")


import golois

# ───────────────────────────────────────────────────────────────────
# CONSTANTES GLOBALES
# ───────────────────────────────────────────────────────────────────
DRIVE_DIR  = '/kaggle/working/go_project3'
RUNS_DIR   = 'runs'
PLANES     = 31
MOVES      = 361
N          = 50000

os.makedirs(DRIVE_DIR, exist_ok=True)
os.makedirs(RUNS_DIR,  exist_ok=True)

print(f"TensorFlow : {tf.__version__}")
print(f"GPUs       : {tf.config.list_physical_devices('GPU')}")

# ───────────────────────────────────────────────────────────────────
# SEEDS
# ───────────────────────────────────────────────────────────────────
def set_seeds(seed):
    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)
    os.environ['PYTHONHASHSEED']       = str(seed)
    os.environ['TF_DETERMINISTIC_OPS'] = '1'

set_seeds(42)
# ───────────────────────────────────────────────────────────────────
# DONNÉES — init une seule fois, réutilisées par tous les modèles
# ───────────────────────────────────────────────────────────────────
print("\nInitialisation des données...", flush=True)

# Train (écrasé à chaque getBatch)
input_data = np.random.randint(2, size=(N, 19, 19, PLANES)).astype('float32')
policy     = keras.utils.to_categorical(
                np.random.randint(MOVES, size=(N,)), num_classes=MOVES
             ).astype('float32')
value      = np.random.randint(2, size=(N,)).astype('float32')
end        = np.random.randint(2, size=(N, 19, 19, 2)).astype('float32')
groups     = np.zeros((N, 19, 19, 1), dtype='float32')

# Val set fixe — rempli une seule fois, jamais retouché
val_input  = np.zeros((N, 19, 19, PLANES), dtype='float32')
val_policy = np.zeros((N, MOVES),          dtype='float32')
val_value  = np.zeros((N,),                dtype='float32')
val_end    = np.zeros((N, 19, 19, 2),      dtype='float32')

golois.getValidation(val_input, val_policy, val_value, val_end)
print("✅ Val set fixe chargé — ne sera plus modifié\n")

# ───────────────────────────────────────────────────────────────────
# HELPERS
# ───────────────────────────────────────────────────────────────────
def make_run_dir(config):
    """Crée le dossier de run horodaté et écrit config.txt."""
    timestamp = datetime.datetime.now().strftime('%Y%m%d_%H%M%S')
    run_name  = (f"model{config['model_id']}_{config['model_name']}"
                 f"_lr{config['lr']}_seed{config['seed']}_{timestamp}")
    run_dir   = os.path.join(RUNS_DIR, run_name)
    os.makedirs(run_dir, exist_ok=True)

    with open(os.path.join(run_dir, 'config.txt'), 'w') as f:
        for k, v in config.items():
            f.write(f"{k} = {v}\n")
        f.write(f"\ntf_version = {tf.__version__}\n")
        f.write(f"gpu        = {tf.config.list_physical_devices('GPU')}\n")
        f.write(f"timestamp  = {timestamp}\n")

    print(f"\n{'='*60}")
    print(f"  RUN : {run_name}")
    print(f"  DIR : {run_dir}")
    print(f"{'='*60}\n")
    return run_dir, run_name, timestamp


def make_csv(run_dir):
    """Crée les fichiers CSV train et val avec leurs headers."""
    csv_fields = ['epoch', 'loss', 'policy_loss', 'value_loss',
                  'policy_categorical_accuracy', 'value_mae']
    val_fields = ['epoch', 'total_loss', 'policy_loss', 'value_loss',
                  'policy_categorical_accuracy', 'value_mae']

    csv_path = os.path.join(run_dir, 'history.csv')
    val_path = os.path.join(run_dir, 'val_results.csv')

    with open(csv_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=csv_fields).writeheader()
    with open(val_path, 'w', newline='') as f:
        csv.DictWriter(f, fieldnames=val_fields).writeheader()

    return csv_path, csv_fields, val_path, val_fields


def plot_curves(history_log, run_dir, run_name, epoch):
    """Sauvegarde les courbes train intermédiaires."""
    epochs_range = history_log['epoch']
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    fig.suptitle(run_name, fontsize=9)

    axes[0].plot(epochs_range, history_log['loss'], 'b-o', markersize=3)
    axes[0].set_title('Total Loss'); axes[0].set_xlabel('Epoch'); axes[0].grid(True, alpha=0.3)

    axes[1].plot(epochs_range, history_log['policy_categorical_accuracy'], 'g-o', markersize=3)
    axes[1].set_title('Policy Accuracy'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)

    axes[2].plot(epochs_range, history_log['value_mae'], 'r-o', markersize=3)
    axes[2].set_title('Value MAE'); axes[2].set_xlabel('Epoch'); axes[2].grid(True, alpha=0.3)

    plt.tight_layout()
    plot_path = os.path.join(run_dir, f'curves_epoch{epoch}.png')
    plt.savefig(plot_path, dpi=120)
    plt.close()
    print(f"  [PLOT] → {plot_path}")


def save_final_plots(history_log, run_dir):
    """Sauvegarde les graphiques rapport final."""
    plt.style.use('seaborn-v0_8-whitegrid')
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))
    epochs = history_log['epoch']

    ax1.plot(epochs, history_log['policy_categorical_accuracy'],
             color='#2ca02c', label='Policy Accuracy')
    ax1.set_title('Évolution de la précision (Policy)')
    ax1.set_xlabel('Époques'); ax1.set_ylabel('Précision'); ax1.legend()

    ax2.plot(epochs, history_log['loss'], color='#d62728', label='Total Loss')
    ax2.set_title('Évolution de la perte (Loss)')
    ax2.set_xlabel('Époques'); ax2.set_ylabel('Perte'); ax2.legend()

    plt.tight_layout()
    plt.savefig(os.path.join(run_dir, 'report_visuals.png'), dpi=300)
    plt.close()
    print("✅ report_visuals.png sauvegardé")


def drive_sync_light(run_dir, run_name):
    """Sync léger en cours de run : CSV + PNG + TXT uniquement."""
    drive_run_dir = os.path.join(DRIVE_DIR, run_name)
    os.makedirs(drive_run_dir, exist_ok=True)
    for fname in os.listdir(run_dir):
        if fname.endswith(('.csv', '.png', '.txt')):
            shutil.copy(os.path.join(run_dir, fname),
                        os.path.join(drive_run_dir, fname))
    print(f"  [DRIVE] Sync léger → {drive_run_dir}")


def drive_sync_full(run_dir, run_name):
    """Sync complet en fin de run (inclut les .keras)."""
    drive_run_dir = os.path.join(DRIVE_DIR, run_name)
    shutil.copytree(run_dir, drive_run_dir, dirs_exist_ok=True)

    global_csv = os.path.join(RUNS_DIR, 'all_runs_summary.csv')
    if os.path.exists(global_csv):
        shutil.copy(global_csv, os.path.join(DRIVE_DIR, 'all_runs_summary.csv'))
    print(f"  [DRIVE] Sync final → {drive_run_dir}")


def update_global_summary(config, run_name, timestamp,
                           total_params, best_loss, best_policy_acc, epochs_run):
    """Ajoute une ligne au CSV global all_runs_summary.csv."""
    global_csv    = os.path.join(RUNS_DIR, 'all_runs_summary.csv')
    global_fields = ['run_name', 'model_id', 'model_name', 'params',
                     'best_loss', 'best_policy_acc',
                     'lr', 'seed', 'epochs_run', 'timestamp']
    file_exists = os.path.exists(global_csv)

    with open(global_csv, 'a', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=global_fields)
        if not file_exists:
            writer.writeheader()
        writer.writerow({
            'run_name'        : run_name,
            'model_id'        : config['model_id'],
            'model_name'      : config['model_name'],
            'params'          : total_params,
            'best_loss'       : round(best_loss, 6),
            'best_policy_acc' : round(best_policy_acc, 6),
            'lr'              : config['lr'],
            'seed'            : config['seed'],
            'epochs_run'      : epochs_run,
            'timestamp'       : timestamp,
        })


def print_final_summary(config, run_dir, run_name,
                         best_loss, best_policy_acc, total_params, epochs_run):
    print(f"\n{'='*60}")
    print(f"  RUN TERMINÉ : {run_name}")
    print(f"  Drive       : {DRIVE_DIR}/{run_name}")
    print(f"  Best loss        : {best_loss:.4f}")
    print(f"  Best policy acc  : {best_policy_acc:.4f}")
    print(f"  Epochs réelles   : {epochs_run}/{config['epochs']}")
    print(f"  Params           : {total_params:,}")
    print(f"  Outputs          : {run_dir}/")
    print(f"    ├── config.txt")
    print(f"    ├── history.csv          (train, toutes les epochs)")
    print(f"    ├── val_results.csv      (val fixe, tous les 10 epochs)")
    print(f"    ├── best_model.keras")
    print(f"    ├── logs/                (TensorBoard)")
    print(f"    ├── curves_epoch[n].png  (plots intermédiaires)")
    print(f"    └── report_visuals.png   (graphiques rapport)")
    print(f"  Global : {RUNS_DIR}/all_runs_summary.csv")
    print(f"{'='*60}\n")

# ───────────────────────────────────────────────────────────────────
# FONCTION D'ENTRAÎNEMENT UNIVERSELLE
# ───────────────────────────────────────────────────────────────────
def train_model(build_fn, config):
    """
    Entraîne n'importe quel modèle compilé avec le socle standard.
    build_fn(config) est appelé dans strategy.scope().
    """
    set_seeds(config['seed'])
    with strategy.scope():
        model = build_fn(config)

    # ── Setup run ────────────────────────────────────────────────────
    run_dir, run_name, timestamp = make_run_dir(config)
    wandb.init(
        project = "go-19x19",
        name    = run_name,
        config  = config,
        reinit  = True
    )
    csv_path, csv_fields, val_path, val_fields = make_csv(run_dir)
    history_log = {k: [] for k in csv_fields}

    checkpoint_path = os.path.join(run_dir, 'best_model.keras')
    log_dir         = os.path.join(run_dir, 'logs')

    # ── Callbacks ────────────────────────────────────────────────────
    callbacks = [
        keras.callbacks.ReduceLROnPlateau(
            monitor='loss', factor=0.5,
            patience=config['patience'], mode='min',
            min_lr=1e-6, min_delta=1e-4,
            verbose=1,
        ),
    ]
    tb_writer = tf.summary.create_file_writer(log_dir)

    # ── Boucle ───────────────────────────────────────────────────────
    best_loss          = float('inf')
    best_policy_acc    = 0.0
    early_stop_counter = 0

    for i in range(1, config['epochs'] + 1):
        print(f"\n{'─'*50}")
        print(f"  Epoch {i}/{config['epochs']}")
        print(f"{'─'*50}")

        golois.getBatch(input_data, policy, value, end, groups, i * N)

        hist = model.fit(
            input_data, [policy, value],
            initial_epoch=i - 1,
            epochs=i,
            batch_size=config['batch'],
            callbacks=callbacks,
            verbose=1
        )

        # ── Métriques ────────────────────────────────────────────────
        h = hist.history
        row = {
            'epoch'                       : i,
            'loss'                        : h['loss'][0],
            'policy_loss'                 : h['policy_loss'][0],
            'value_loss'                  : h['value_loss'][0],
            'policy_categorical_accuracy' : h['policy_categorical_accuracy'][0],
            'value_mae'                   : h['value_mae'][0],
        }

        with open(csv_path, 'a', newline='') as f:
            csv.DictWriter(f, fieldnames=csv_fields).writerow(row)
        for k, v in row.items():
            history_log[k].append(v)

        if row['policy_categorical_accuracy'] > best_policy_acc:
            best_policy_acc = row['policy_categorical_accuracy']

        print(f"  loss={row['loss']:.4f} | "
              f"policy_acc={row['policy_categorical_accuracy']:.4f} | "
              f"value_mae={row['value_mae']:.4f}")

        # ── TensorBoard ──────────────────────────────────────────────
        with tb_writer.as_default():
            tf.summary.scalar('loss/total',  row['loss'],                        step=i)
            tf.summary.scalar('loss/policy', row['policy_loss'],                 step=i)
            tf.summary.scalar('loss/value',  row['value_loss'],                  step=i)
            tf.summary.scalar('acc/policy',  row['policy_categorical_accuracy'], step=i)
            tf.summary.scalar('mae/value',   row['value_mae'],                   step=i)
        tb_writer.flush()

        # ── Val loss (chaque epoch — pour checkpoint et early stopping) ─
        val = model.evaluate(
            val_input, [val_policy, val_value],
            verbose=0, batch_size=config['batch']
        )

        current_val_loss       = val[0]
        current_val_policy_acc = val[3]

        wandb.log({
            'epoch'            : i,
            'train_loss'       : row['loss'],
            'train_policy_acc' : row['policy_categorical_accuracy'],
            'train_value_mae'  : row['value_mae'],
            'val_loss'         : current_val_loss,
            'val_policy_acc'   : current_val_policy_acc,
            'lr'               : float(keras.backend.get_value(
                                     model.optimizer.learning_rate))
        })

        # ── Early stopping ───────────────────────────────────────────────
        if current_val_loss < best_loss:
            best_loss          = current_val_loss
            early_stop_counter = 0
            model.save(checkpoint_path)  # ← save manuel basé sur val_loss
            print(f"  [CHECKPOINT] val_loss → {best_loss:.4f} — modèle sauvegardé")
        else:
            early_stop_counter += 1
            print(f"  [EarlyStopping] patience {early_stop_counter}/{config['patience']}")
            if early_stop_counter >= config['patience']:
                print(f"  [EarlyStopping] Arrêt à l'époque {i}.")
                break

        if i % 5 == 0:
            gc.collect()

        # ── Val + plot + sync tous les 10 epochs ─────────────────────
        if i % 10 == 0:
            print(f"\n  [VAL epoch {i}] total={current_val_loss:.4f} | "
                  f"policy_acc={val[3]:.4f} | value_mae={val[4]:.4f}")

            with open(val_path, 'a', newline='') as f:
                csv.DictWriter(f, fieldnames=val_fields).writerow({
                    'epoch'                       : i,
                    'total_loss'                  : current_val_loss,
                    'policy_loss'                 : val[1],
                    'value_loss'                  : val[2],
                    'policy_categorical_accuracy' : val[3],
                    'value_mae'                   : val[4],
                })

            plot_curves(history_log, run_dir, run_name, i)
            drive_sync_light(run_dir, run_name)

    tb_writer.close()

    # ── Fin de run ───────────────────────────────────────────────────
    epochs_run = len(history_log['epoch'])
    save_final_plots(history_log, run_dir)
    update_global_summary(config, run_name, timestamp,
                          model.count_params(), best_loss, best_policy_acc, epochs_run)
    drive_sync_full(run_dir, run_name)
    print_final_summary(config, run_dir, run_name,
                        best_loss, best_policy_acc, model.count_params(), epochs_run)
    wandb.finish()
    return history_log, best_loss, best_policy_acc


print("✅ Cellule BASE chargée — prêt pour les modèles")

GPUs détectés : 1
TensorFlow : 2.19.0
GPUs       : [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Initialisation des données...
✅ Val set fixe chargé — ne sera plus modifié

✅ Cellule BASE chargée — prêt pour les modèles


# 11

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M11 — MBConv + SE Attention | Projet Go 19×19
#  MobileNetV2-style : expand(×2) → depthwise → SE → project
#  9 blocs, 36 filtres, activation Swish, stochastic depth
#  ≈ 98 242 params  — 0 Lambda layer
#
#  Budget :
#    Stem  Conv3×3(36) + BN         =  10 188
#    9 × MBConv-SE                  =  83 106   (9 234 / bloc)
#      dont : expand  36→72 + BN   =   2 880
#             DW 3×3        + BN   =     936
#             SE  72→18→72  (bias) =   2 466
#             project 72→36 + BN   =   2 736  (+ résidu gratuit)
#    Policy head  Conv1×1(1) + flat =      37
#    Value head   GAP→Dense(192)→1 =   7 297
#    ─────────────────────────────────────────
#    Total                           =  98 242  ✓ < 100 000
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 11,
    'model_name' : 'mbconv_se',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 36,
    'num_blocks' : 9,
    'expand'     : 2,
    'se_ratio'   : 4,
    'drop_rate'  : 0.05,
}


class StochasticDepth(keras.layers.Layer):
    def __init__(self, drop_rate=0.05, **kwargs):
        super().__init__(**kwargs)
        self.drop_rate = drop_rate

    def call(self, x, training=None):
        if not training or self.drop_rate == 0.0:
            return x
        keep_prob = 1.0 - self.drop_rate
        shape = (tf.shape(x)[0], 1, 1, 1)
        noise = tf.floor(keep_prob + tf.random.uniform(shape, dtype=x.dtype))
        return x * noise / keep_prob

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'drop_rate': self.drop_rate})
        return cfg


def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    nb  = config['num_blocks']
    ex  = config['expand']
    sr  = config['se_ratio']
    dr  = config['drop_rate']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    def mbconv_se(x_in, name):
        C     = x_in.shape[-1]
        exp_C = C * ex
        se_C  = exp_C // sr

        h = layers.Conv2D(exp_C, (1, 1), use_bias=False,
                          kernel_regularizer=reg,
                          name=f'{name}_expand')(x_in)
        h = layers.BatchNormalization(name=f'{name}_bn1')(h)
        h = layers.Activation('swish', name=f'{name}_act1')(h)

        h = layers.DepthwiseConv2D((3, 3), padding='same', use_bias=False,
                                    depthwise_regularizer=reg,
                                    name=f'{name}_dw')(h)
        h = layers.BatchNormalization(name=f'{name}_bn2')(h)
        h = layers.Activation('swish', name=f'{name}_act2')(h)

        se = layers.GlobalAveragePooling2D(name=f'{name}_se_gap')(h)
        se = layers.Dense(se_C, activation='relu',
                          kernel_regularizer=reg,
                          name=f'{name}_se_fc1')(se)
        se = layers.Dense(exp_C, activation='sigmoid',
                          kernel_regularizer=reg,
                          name=f'{name}_se_fc2')(se)
        se = layers.Reshape((1, 1, exp_C), name=f'{name}_se_rs')(se)
        h  = layers.Multiply(name=f'{name}_se_mul')([h, se])

        h = layers.Conv2D(C, (1, 1), use_bias=False,
                          kernel_regularizer=reg,
                          name=f'{name}_proj')(h)
        h = layers.BatchNormalization(name=f'{name}_bn3')(h)

        if dr > 0.0:
            h = StochasticDepth(drop_rate=dr, name=f'{name}_sd')(h)
        return layers.Add(name=f'{name}_add')([x_in, h])

    for i in range(nb):
        x = mbconv_se(x, name=f'b{i + 1}')

    p = layers.Conv2D(1, (1, 1), kernel_regularizer=reg,
                      name='policy_conv')(x)
    p = layers.Flatten(name='policy_flat')(p)
    policy_head = layers.Activation('softmax', name='policy',
                                     dtype='float32')(p)

    v = layers.GlobalAveragePooling2D(name='value_gap')(x)
    v = layers.Dense(192, activation='swish',
                     kernel_regularizer=reg, name='value_fc')(v)
    value_head = layers.Dense(1, activation='tanh',
                               kernel_regularizer=reg,
                               name='value', dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr'],
                                        clipnorm=1.0),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model


history_m11, best_loss_m11, best_acc_m11 = train_model(build_model, config)

# 12

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M12 — Inverted Residual + ECA | Projet Go 19×19
#  MobileNetV2-style : expand(×3) → depthwise → ECA → project
#  7 blocs, 38 filtres, activation Swish, stochastic depth
#  ≈ 93 773 params  — 0 Lambda layer
#
#  Budget :
#    Stem  Conv3×3(38) + BN          =  10 754
#    7 × InvRes-ECA                  =  75 299   (10 757 / bloc)
#      dont : expand  38→114 + BN    =   5 472
#             DW 3×3         + BN    =   5 472
#             ECA Conv1D(ks=3)        =       3
#             project 114→38 + BN    =   4 484
#    Policy head  Conv1×1(1) + flat  =      39
#    Value head   GAP→Dense(192)→1  =   7 681
#    ─────────────────────────────────────────
#    Total                            =  93 773  ✓ < 100 000
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 12,
    'model_name' : 'inverted_eca',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 38,
    'num_blocks' : 7,
    'expand'     : 3,
    'drop_rate'  : 0.05,
}


class StochasticDepth(keras.layers.Layer):
    def __init__(self, drop_rate=0.05, **kwargs):
        super().__init__(**kwargs)
        self.drop_rate = drop_rate

    def call(self, x, training=None):
        if not training or self.drop_rate == 0.0:
            return x
        keep_prob = 1.0 - self.drop_rate
        shape = (tf.shape(x)[0], 1, 1, 1)
        noise = tf.floor(keep_prob + tf.random.uniform(shape, dtype=x.dtype))
        return x * noise / keep_prob

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'drop_rate': self.drop_rate})
        return cfg


def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    nb  = config['num_blocks']
    ex  = config['expand']
    dr  = config['drop_rate']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    def eca_block(h, name):
        C = h.shape[-1]
        g = layers.GlobalAveragePooling2D(keepdims=True,
                                           name=f'{name}_gap')(h)
        g = layers.Reshape((C, 1), name=f'{name}_rs_in')(g)
        g = layers.Conv1D(1, kernel_size=3, padding='same',
                           use_bias=False, name=f'{name}_conv')(g)
        g = layers.Activation('sigmoid', name=f'{name}_sig')(g)
        g = layers.Reshape((1, 1, C), name=f'{name}_rs_out')(g)
        return layers.Multiply(name=f'{name}_mul')([h, g])

    def inv_res_eca(x_in, name):
        C     = x_in.shape[-1]
        exp_C = C * ex

        h = layers.Conv2D(exp_C, (1, 1), use_bias=False,
                          kernel_regularizer=reg,
                          name=f'{name}_expand')(x_in)
        h = layers.BatchNormalization(name=f'{name}_bn1')(h)
        h = layers.Activation('swish', name=f'{name}_act1')(h)

        h = layers.DepthwiseConv2D((3, 3), padding='same', use_bias=False,
                                    depthwise_regularizer=reg,
                                    name=f'{name}_dw')(h)
        h = layers.BatchNormalization(name=f'{name}_bn2')(h)
        h = layers.Activation('swish', name=f'{name}_act2')(h)

        h = eca_block(h, name=f'{name}_eca')

        h = layers.Conv2D(C, (1, 1), use_bias=False,
                          kernel_regularizer=reg,
                          name=f'{name}_proj')(h)
        h = layers.BatchNormalization(name=f'{name}_bn3')(h)

        if dr > 0.0:
            h = StochasticDepth(drop_rate=dr, name=f'{name}_sd')(h)
        return layers.Add(name=f'{name}_add')([x_in, h])

    for i in range(nb):
        x = inv_res_eca(x, name=f'b{i + 1}')

    p = layers.Conv2D(1, (1, 1), kernel_regularizer=reg,
                      name='policy_conv')(x)
    p = layers.Flatten(name='policy_flat')(p)
    policy_head = layers.Activation('softmax', name='policy',
                                     dtype='float32')(p)

    v = layers.GlobalAveragePooling2D(name='value_gap')(x)
    v = layers.Dense(192, activation='swish',
                     kernel_regularizer=reg, name='value_fc')(v)
    value_head = layers.Dense(1, activation='tanh',
                               kernel_regularizer=reg,
                               name='value', dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr'],
                                        clipnorm=1.0),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model


history_m12, best_loss_m12, best_acc_m12 = train_model(build_model, config)

# 15

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M15 — ConvNeXt-Lite | Projet Go 19×19
#  Depthwise 7×7 (ConvNeXt-style) + expand×4 + ECA
#  7 blocs, 26 filtres, activation Swish, stochastic depth
#  ≈ 92 863 params — 0 Lambda layer
#
#  Budget :
#    Stem  Conv3×3(26) + BN           =   7 358
#    7 × ConvNeXt block               =  80 101   (11 443 / bloc)
#      dont : DW 7×7          + BN    =   5 512   ← clé de l'archi
#             expand  26→104  + BN    =   3 120
#             ECA Conv1D(ks=3)         =       3
#             project 104→26  + BN    =   2 808
#    Policy head  Conv1×1(1) + flat   =      27
#    Value head   GAP→Dense(192)→1   =   5 377
#    ─────────────────────────────────────────
#    Total                             =  92 863  ✓ < 100 000
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 15,
    'model_name' : 'convnext_lite',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 26,
    'num_blocks' : 7,
    'expand'     : 4,
    'dw_kernel'  : 7,
    'drop_rate'  : 0.05,
}


class StochasticDepth(keras.layers.Layer):
    def __init__(self, drop_rate=0.05, **kwargs):
        super().__init__(**kwargs)
        self.drop_rate = drop_rate

    def call(self, x, training=None):
        if not training or self.drop_rate == 0.0:
            return x
        keep_prob = 1.0 - self.drop_rate
        shape = (tf.shape(x)[0], 1, 1, 1)
        noise = tf.floor(keep_prob + tf.random.uniform(shape, dtype=x.dtype))
        return x * noise / keep_prob

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'drop_rate': self.drop_rate})
        return cfg


def build_model(config):
    reg = regularizers.l2(config['l2'])
    f   = config['filters']
    nb  = config['num_blocks']
    ex  = config['expand']
    ks  = config['dw_kernel']
    dr  = config['drop_rate']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    def eca_block(h, name):
        C = h.shape[-1]
        g = layers.GlobalAveragePooling2D(keepdims=True,
                                           name=f'{name}_gap')(h)
        g = layers.Reshape((C, 1), name=f'{name}_rs_in')(g)
        g = layers.Conv1D(1, kernel_size=3, padding='same',
                           use_bias=False, name=f'{name}_conv')(g)
        g = layers.Activation('sigmoid', name=f'{name}_sig')(g)
        g = layers.Reshape((1, 1, C), name=f'{name}_rs_out')(g)
        return layers.Multiply(name=f'{name}_mul')([h, g])

    def convnext_block(x_in, name):
        C     = x_in.shape[-1]
        exp_C = C * ex

        h = layers.DepthwiseConv2D((ks, ks), padding='same', use_bias=False,
                                    depthwise_regularizer=reg,
                                    name=f'{name}_dw')(x_in)
        h = layers.BatchNormalization(name=f'{name}_bn_dw')(h)
        h = layers.Activation('swish', name=f'{name}_act_dw')(h)

        h = layers.Conv2D(exp_C, (1, 1), use_bias=False,
                          kernel_regularizer=reg,
                          name=f'{name}_expand')(h)
        h = layers.BatchNormalization(name=f'{name}_bn_exp')(h)
        h = layers.Activation('swish', name=f'{name}_act_exp')(h)

        h = eca_block(h, name=f'{name}_eca')

        h = layers.Conv2D(C, (1, 1), use_bias=False,
                          kernel_regularizer=reg,
                          name=f'{name}_proj')(h)
        h = layers.BatchNormalization(name=f'{name}_bn_proj')(h)

        if dr > 0.0:
            h = StochasticDepth(drop_rate=dr, name=f'{name}_sd')(h)
        return layers.Add(name=f'{name}_add')([x_in, h])

    for i in range(nb):
        x = convnext_block(x, name=f'b{i + 1}')

    p = layers.Conv2D(1, (1, 1), kernel_regularizer=reg,
                      name='policy_conv')(x)
    p = layers.Flatten(name='policy_flat')(p)
    policy_head = layers.Activation('softmax', name='policy',
                                     dtype='float32')(p)

    v = layers.GlobalAveragePooling2D(name='value_gap')(x)
    v = layers.Dense(192, activation='swish',
                     kernel_regularizer=reg, name='value_fc')(v)
    value_head = layers.Dense(1, activation='tanh',
                               kernel_regularizer=reg,
                               name='value', dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr'],
                                        clipnorm=1.0),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model


history_m15, best_loss_m15, best_acc_m15 = train_model(build_model, config)

# 18

In [ ]:
# ═══════════════════════════════════════════════════════════════════
#  CELLULE M18 — EfficientFormer-Go | Projet Go 19×19
#  Staged : 9 MBConv-SE (features locales) → 2 Transformer (global)
#  32 filtres, ≈ 99 666 params — 0 Lambda layer
#
#  Budget :
#    Stem  Conv3×3(32) + BN              =   9 056
#    9 × MBConv-SE  (expand×2, SE/4)    =  66 960   (7 440 / bloc)
#    2 × TransformerBlock (d=32, h=2)   =  17 088   (8 544 / bloc)
#      dont : MHA  (2 têtes, key=16)    =   4 224
#             FFN  Dense(64) + Dense(32) =   4 192
#             LayerNorm × 2              =     128
#    Policy head  Conv1×1(1) + flat     =      33
#    Value head   GAP→Dense(192)→1     =   6 529
#    ─────────────────────────────────────────
#    Total                               =  99 666  ✓ < 100 000
#
#  ⚠️  Nécessite la cellule BASE exécutée au préalable
# ═══════════════════════════════════════════════════════════════════

config = {
    'model_id'   : 18,
    'model_name' : 'efficient_former',
    'planes'     : PLANES,
    'moves'      : MOVES,
    'N'          : N,
    'epochs'     : 300,
    'batch'      : 128 * NUM_GPUS,
    'lr'         : 0.001,
    'l2'         : 0.0001,
    'seed'       : 42,
    'max_params' : 100_000,
    'patience'   : 20,
    'filters'    : 32,
    'num_blocks' : 9,
    'expand'     : 2,
    'se_ratio'   : 4,
    'trans_heads': 2,
    'trans_ffn'  : 64,
    'drop_rate'  : 0.05,
}


class StochasticDepth(keras.layers.Layer):
    def __init__(self, drop_rate=0.05, **kwargs):
        super().__init__(**kwargs)
        self.drop_rate = drop_rate

    def call(self, x, training=None):
        if not training or self.drop_rate == 0.0:
            return x
        keep_prob = 1.0 - self.drop_rate
        shape = (tf.shape(x)[0], 1, 1, 1)
        noise = tf.floor(keep_prob + tf.random.uniform(shape, dtype=x.dtype))
        return x * noise / keep_prob

    def get_config(self):
        cfg = super().get_config()
        cfg.update({'drop_rate': self.drop_rate})
        return cfg


def build_model(config):
    reg    = regularizers.l2(config['l2'])
    f      = config['filters']
    nb     = config['num_blocks']
    ex     = config['expand']
    sr     = config['se_ratio']
    dr     = config['drop_rate']
    heads  = config['trans_heads']
    ffn    = config['trans_ffn']

    inp = keras.Input(shape=(19, 19, config['planes']), name='board')

    x = layers.Conv2D(f, (3, 3), padding='same', use_bias=False,
                      kernel_regularizer=reg)(inp)
    x = layers.BatchNormalization()(x)
    x = layers.Activation('swish')(x)

    def mbconv_se(x_in, name):
        C = x_in.shape[-1]; exp_C = C * ex; se_C = exp_C // sr
        h = layers.Conv2D(exp_C, (1, 1), use_bias=False,
                          kernel_regularizer=reg, name=f'{name}_expand')(x_in)
        h = layers.BatchNormalization(name=f'{name}_bn1')(h)
        h = layers.Activation('swish', name=f'{name}_act1')(h)
        h = layers.DepthwiseConv2D((3, 3), padding='same', use_bias=False,
                                    depthwise_regularizer=reg,
                                    name=f'{name}_dw')(h)
        h = layers.BatchNormalization(name=f'{name}_bn2')(h)
        h = layers.Activation('swish', name=f'{name}_act2')(h)
        se = layers.GlobalAveragePooling2D(name=f'{name}_se_gap')(h)
        se = layers.Dense(se_C, activation='relu',
                          kernel_regularizer=reg,
                          name=f'{name}_se_fc1')(se)
        se = layers.Dense(exp_C, activation='sigmoid',
                          kernel_regularizer=reg,
                          name=f'{name}_se_fc2')(se)
        se = layers.Reshape((1, 1, exp_C), name=f'{name}_se_rs')(se)
        h  = layers.Multiply(name=f'{name}_se_mul')([h, se])
        h = layers.Conv2D(C, (1, 1), use_bias=False,
                          kernel_regularizer=reg, name=f'{name}_proj')(h)
        h = layers.BatchNormalization(name=f'{name}_bn3')(h)
        if dr > 0.0:
            h = StochasticDepth(drop_rate=dr, name=f'{name}_sd')(h)
        return layers.Add(name=f'{name}_add')([x_in, h])

    def transformer_block(x_in, name):
        attn = layers.MultiHeadAttention(
            num_heads=heads, key_dim=f // heads,
            name=f'{name}_mha')(x_in, x_in)
        x = layers.Add(name=f'{name}_add1')([x_in, attn])
        x = layers.LayerNormalization(name=f'{name}_ln1')(x)
        ff = layers.Dense(ffn, activation='gelu', name=f'{name}_ffn1')(x)
        ff = layers.Dense(f, name=f'{name}_ffn2')(ff)
        x  = layers.Add(name=f'{name}_add2')([x, ff])
        x  = layers.LayerNormalization(name=f'{name}_ln2')(x)
        return x

    for i in range(nb):
        x = mbconv_se(x, name=f'b{i + 1}')

    x = layers.Reshape((19 * 19, f), name='to_seq')(x)
    x = transformer_block(x, name='trans1')
    x = transformer_block(x, name='trans2')
    x = layers.Reshape((19, 19, f), name='to_spatial')(x)

    p = layers.Conv2D(1, (1, 1), kernel_regularizer=reg,
                      name='policy_conv')(x)
    p = layers.Flatten(name='policy_flat')(p)
    policy_head = layers.Activation('softmax', name='policy',
                                     dtype='float32')(p)

    v = layers.GlobalAveragePooling2D(name='value_gap')(x)
    v = layers.Dense(192, activation='swish',
                     kernel_regularizer=reg, name='value_fc')(v)
    value_head = layers.Dense(1, activation='tanh',
                               kernel_regularizer=reg,
                               name='value', dtype='float32')(v)

    model = keras.Model(inputs=inp, outputs=[policy_head, value_head])
    model.summary()

    total_params = model.count_params()
    print(f"\n>>> Total params : {total_params:,}")
    assert total_params < config['max_params'], \
        f"CONTRAINTE VIOLÉE : {total_params:,} > {config['max_params']:,} !"
    print(f">>> Contrainte < {config['max_params']:,} params : ✓\n")

    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=config['lr'],
                                        clipnorm=1.0),
        loss={'policy': 'categorical_crossentropy', 'value': 'mse'},
        loss_weights={'policy': 1.0, 'value': 1.0},
        metrics={'policy': 'categorical_accuracy', 'value': 'mae'}
    )
    return model


history_m18, best_loss_m18, best_acc_m18 = train_model(build_model, config)